# Profiling — SINAN Dengue 2022–2024

Executa o [roteiro de profiling](ROTEIRO.md) e gera `profiling/RESULTADOS.md` preenchido.

**No Colab:** *Ambiente de execução → Executar tudo*. O notebook monta o Google Drive e grava
tudo em `MyDrive/dengue-comorbidity-risk/` (dados, manifesto e `profiling/RESULTADOS.md`).
Na primeira execução, o download e a conversão de 2024 levam alguns minutos; nas seguintes,
os Parquets são reaproveitados se o arquivo do DataSUS não mudou.

**Local:** `uv run jupyter nbconvert --to notebook --execute --inplace profiling/01_profiling.ipynb`

Fonte: FTP oficial do DataSUS (D-05). Nenhum microdado sai do Drive/disco local; o
`RESULTADOS.md` só tem contagens agregadas.

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Mesmas versões do uv.lock (D-16).
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "pyreaddbc==2.0.4", "dbfread==2.0.7"])
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/dengue-comorbidity-risk")
else:
    BASE = Path.cwd().parent if Path.cwd().name == "profiling" else Path.cwd()

DATA_DIR = BASE / "data"
OUT_MD = BASE / "profiling" / "RESULTADOS.md"
ANOS = [2022, 2023, 2024]
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_MD.parent.mkdir(parents=True, exist_ok=True)
print("Base:", BASE)

## Código (copiado de `src/dengue_sinan`)

In [ ]:
# Copiado de src/dengue_sinan/features.py — não edite aqui; edite o módulo e regere.
"""Transformações de campos do SINAN em variáveis analisáveis."""

import numpy as np
import pandas as pd

# NU_IDADE_N: 1º dígito = unidade, 3 dígitos seguintes = valor (dicionário NINDIN, §3.7 do spec).
_AGE_UNIT_TO_YEARS = {
    1: 1 / 24 / 365.25,  # horas
    2: 1 / 365.25,  # dias
    3: 1 / 12,  # meses
    4: 1.0,  # anos
}

# 1º dígito do código IBGE de UF/município → macrorregião.
_IBGE_REGION = {
    "1": "Norte",
    "2": "Nordeste",
    "3": "Sudeste",
    "4": "Sul",
    "5": "Centro-Oeste",
}


def _as_code_string(series: pd.Series) -> pd.Series:
    """Normaliza códigos que chegam como int, float ("4018.0") ou texto."""
    text = series.astype("string").str.strip()
    return text.str.replace(r"\.0$", "", regex=True)


def decode_age_years(nu_idade_n: pd.Series) -> pd.Series:
    """Converte NU_IDADE_N para anos fracionários; códigos inválidos viram NaN."""
    code = _as_code_string(nu_idade_n)
    valid = code.str.fullmatch(r"[1-4]\d{3}").fillna(False).astype(bool)
    unit = pd.to_numeric(code.where(valid).str[0], errors="coerce")
    value = pd.to_numeric(code.where(valid).str[1:], errors="coerce")
    factor = unit.map(_AGE_UNIT_TO_YEARS)
    return (value * factor).astype("float64").where(valid, np.nan)


def region_from_ibge(code: pd.Series) -> pd.Series:
    """Macrorregião a partir do código IBGE de UF (2 dígitos) ou município (6–7 dígitos)."""
    text = _as_code_string(code)
    well_formed = text.str.fullmatch(r"[1-5]\d(\d{4,5})?").fillna(False).astype(bool)
    return text.where(well_formed).str[0].map(_IBGE_REGION)

In [ ]:
# Copiado de src/dengue_sinan/download.py — não edite aqui; edite o módulo e regere.
"""Download dos microdados SINAN-Dengue direto do FTP do DataSUS (D-05)."""

import datetime as dt
import ftplib
import hashlib
import json
import platform
import tempfile
from importlib import metadata
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pyreaddbc
from dbfread import DBF

FTP_HOST = "ftp.datasus.gov.br"
# Ordem de busca: arquivo final primeiro; preliminar só se o final não existir.
FTP_DIRS = {
    "final": "/dissemin/publicos/SINAN/DADOS/FINAIS",
    "preliminar": "/dissemin/publicos/SINAN/DADOS/PRELIM",
}
DBF_ENCODING = "iso-8859-1"
CHUNK_ROWS = 200_000


def remote_name(year: int) -> str:
    """Nome do arquivo anual de dengue no FTP (ex.: 2022 → DENGBR22.dbc)."""
    return f"DENGBR{year % 100:02d}.dbc"


def sha256_file(path: Path, block_size: int = 1 << 20) -> str:
    """SHA-256 lendo o arquivo em blocos, sem carregá-lo inteiro na memória."""
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(block_size):
            digest.update(block)
    return digest.hexdigest()


def _locate(ftp: ftplib.FTP, name: str) -> tuple[str, str, int, str]:
    """Acha o arquivo em FINAIS ou PRELIM; devolve (status, dir, tamanho, data de revisão)."""
    for status, directory in FTP_DIRS.items():
        try:
            size = ftp.size(f"{directory}/{name}")
        except ftplib.error_perm:
            continue
        mdtm = ftp.sendcmd(f"MDTM {directory}/{name}").split()[1]
        modified = dt.datetime.strptime(mdtm[:14], "%Y%m%d%H%M%S").replace(tzinfo=dt.UTC)
        return status, directory, size, modified.isoformat()
    raise FileNotFoundError(f"{name} não encontrado em {list(FTP_DIRS.values())}")


def download_dbc(year: int, dest_dir: Path) -> dict:
    """Baixa o .dbc de um ano e devolve os metadados de procedência."""
    name = remote_name(year)
    dest_dir.mkdir(parents=True, exist_ok=True)
    target = dest_dir / name
    with ftplib.FTP(FTP_HOST, timeout=120) as ftp:
        ftp.login()
        ftp.voidcmd("TYPE I")
        status, directory, size, modified = _locate(ftp, name)
        if not (target.exists() and target.stat().st_size == size):
            partial = target.with_suffix(".part")
            with open(partial, "wb") as f:
                ftp.retrbinary(f"RETR {directory}/{name}", f.write, blocksize=1 << 20)
            partial.replace(target)
    if target.stat().st_size != size:
        raise OSError(f"{name}: tamanho local {target.stat().st_size} ≠ remoto {size}")
    return {
        "ano": year,
        "arquivo": name,
        "url": f"ftp://{FTP_HOST}{directory}/{name}",
        "status_datasus": status,
        "revisao_datasus_utc": modified,
        "baixado_em_utc": dt.datetime.now(dt.UTC).isoformat(timespec="seconds"),
        "tamanho_dbc_bytes": size,
        "sha256_dbc": sha256_file(target),
    }


def _column_to_arrow(raw: np.ndarray, length: int) -> pa.Array:
    """Bytes de largura fixa → texto sem espaços nas pontas; vazio vira nulo.

    Caminho rápido para colunas só ASCII (quase todas no SINAN): os bytes viram um array Arrow
    sem cópia, e ASCII é UTF-8 válido. Com algum byte ≥ 0x80, decodifica latin-1 em Python.
    """
    raw = np.ascontiguousarray(raw)
    if (raw.view(np.uint8) < 0x80).all():
        fixed = pa.FixedSizeBinaryArray.from_buffers(
            pa.binary(length), len(raw), [None, pa.py_buffer(raw.view(np.uint8))])
        text = pc.utf8_trim_whitespace(pc.cast(pc.cast(fixed, pa.binary()), pa.string()))
    else:
        text = pa.array([v.decode(DBF_ENCODING).strip() for v in raw.tolist()], type=pa.string())
    return pc.if_else(pc.equal(text, ""), pa.scalar(None, pa.string()), text)


def iter_dbf_tables(dbf_path: Path, chunk_rows: int = CHUNK_ROWS):
    """Lê o DBF em blocos de registros de tamanho fixo (numpy), como tabelas Arrow de texto.

    Equivalente a `dbfread.DBF(raw=True)` (pula registros apagados), mas vetorizado: o
    `dbfread` monta um dict por registro em Python e leva ~6 min só para 2022. Saída conferida
    célula a célula contra o `dbfread` no arquivo de 2022 (1,4 mi linhas × 121 colunas).
    """
    table = DBF(str(dbf_path), raw=True, encoding=DBF_ENCODING)
    header = table.header
    fields = [(f.name, f.length) for f in table.fields]
    dtype = np.dtype([("_deleted", "S1")] + [(name, f"S{length}") for name, length in fields])
    if dtype.itemsize != header.recordlen:
        raise ValueError(f"registro de {header.recordlen} bytes, campos somam {dtype.itemsize}")
    with open(dbf_path, "rb") as f:
        f.seek(header.headerlen)
        remaining = header.numrecords
        while remaining > 0:
            block = np.fromfile(f, dtype=dtype, count=min(chunk_rows, remaining))
            if len(block) == 0:
                raise OSError(f"DBF terminou antes do esperado: faltam {remaining} registros")
            remaining -= len(block)
            block = block[block["_deleted"] != b"*"]
            yield pa.table({name: _column_to_arrow(block[name], length) for name, length in fields})


def dbc_to_parquet(dbc_path: Path, parquet_path: Path, chunk_rows: int = CHUNK_ROWS) -> dict:
    """Converte .dbc → Parquet, todas as colunas como texto (valores crus, sem reinterpretação).

    Grava em blocos, para caber na RAM do Colab mesmo com 2024.
    """
    with tempfile.TemporaryDirectory() as tmp:
        dbf_path = Path(tmp) / dbc_path.with_suffix(".dbf").name
        pyreaddbc.dbc2dbf(str(dbc_path), str(dbf_path))
        header_rows = DBF(str(dbf_path), raw=True).header.numrecords
        parquet_path.parent.mkdir(parents=True, exist_ok=True)
        rows, writer = 0, None
        try:
            for chunk in iter_dbf_tables(dbf_path, chunk_rows):
                if writer is None:
                    writer = pq.ParquetWriter(parquet_path, chunk.schema, compression="zstd")
                    fields = chunk.schema.names
                writer.write_table(chunk)
                rows += chunk.num_rows
        finally:
            if writer is not None:
                writer.close()
    return {
        "linhas": rows,
        "linhas_no_cabecalho_dbf": header_rows,
        "colunas": len(fields),
        "nomes_colunas": fields,
        "sha256_parquet": sha256_file(parquet_path),
    }


def environment_info() -> dict:
    """Versões que entram no manifesto (NFR-02)."""
    packages = ["pyreaddbc", "dbfread", "pyarrow", "pandas"]
    return {
        "python": platform.python_version(),
        **{pkg: metadata.version(pkg) for pkg in packages},
    }


def fetch_year(year: int, data_dir: Path) -> dict:
    """Baixa, converte e registra um ano. Reaproveita o Parquet se o .dbc não mudou."""
    raw_dir, parquet_dir = data_dir / "raw", data_dir / "parquet"
    manifest_path = data_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text("utf-8")) if manifest_path.exists() else {}

    entry = download_dbc(year, raw_dir)
    parquet_path = parquet_dir / f"dengue_{year}.parquet"
    previous = manifest.get(str(year), {})
    reusable = (
        parquet_path.exists()
        and previous.get("sha256_dbc") == entry["sha256_dbc"]
        and previous.get("sha256_parquet") == sha256_file(parquet_path)
    )
    if reusable:
        entry.update({k: previous[k] for k in ("linhas", "linhas_no_cabecalho_dbf", "colunas",
                                               "nomes_colunas", "sha256_parquet")})
    else:
        entry.update(dbc_to_parquet(raw_dir / entry["arquivo"], parquet_path))
    entry["parquet"] = parquet_path.relative_to(data_dir).as_posix()
    entry["ambiente"] = environment_info()

    manifest[str(year)] = entry
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), "utf-8")
    return entry

In [ ]:
# Copiado de src/dengue_sinan/profiling.py — não edite aqui; edite o módulo e regere.
"""Cálculos do roteiro de profiling (profiling/ROTEIRO.md). Funções puras sobre DataFrames.

Convenção: os campos chegam como texto cru do DBF; branco = NaN.
"""

import numpy as np
import pandas as pd


COMORBIDADES = ["DIABETES", "HEMATOLOG", "HEPATOPAT", "RENAL", "HIPERTENSA", "ACIDO_PEPT",
                "AUTO_IMUNE"]
SINAIS = ["FEBRE", "MIALGIA", "CEFALEIA", "EXANTEMA", "VOMITO", "NAUSEA", "DOR_COSTAS",
          "CONJUNTVIT", "ARTRITE", "ARTRALGIA", "PETEQUIA_N", "LEUCOPENIA", "LACO", "DOR_RETRO"]
CAMPOS_ALVO = ["CLASSI_FIN", "EVOLUCAO", "CRITERIO"]
CLASSI_DENGUE = {"10", "11", "12"}
EVOLUCAO_CONHECIDA = {"1", "2"}
GESTANTE_SIM = {"1", "2", "3", "4"}
FAIXAS = [0, 5, 15, 40, 60, 80, np.inf]
FAIXAS_ROTULOS = ["0–4", "5–14", "15–39", "40–59", "60–79", "80+"]
MIN_EVENTOS = 100
BRANCO = "<branco>"


def code(series: pd.Series) -> pd.Series:
    """Texto normalizado: sem espaços e sem sufixo ".0" de números lidos como float."""
    text = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    return text.mask(text == "")


def optional(df: pd.DataFrame, column: str) -> pd.Series:
    """Coluna que pode não existir nos microdados públicos; ausente = toda em branco.

    Ex.: `DT_NASC` não é publicada (anonimização); só `ANO_NASC`. Campos essenciais (alvo,
    comorbidades) devem ser acessados direto, para falhar alto se sumirem.
    """
    if column in df:
        return df[column]
    return pd.Series(pd.NA, index=df.index, dtype="string")


def parse_date(series: pd.Series) -> pd.Series:
    """Datas do DBF (AAAAMMDD) ou ISO; inválidas viram NaT."""
    text = code(series)
    compact = pd.to_datetime(text, format="%Y%m%d", errors="coerce")
    iso = pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")
    return compact.fillna(iso)


def counts_by_year(df: pd.DataFrame, column: str, year: str = "ANO") -> pd.DataFrame:
    """Contagem de cada valor (inclusive branco) por ano. Linhas = valores, colunas = anos."""
    values = code(df[column]).fillna(BRANCO)
    table = pd.crosstab(values, df[year])
    table.index.name = column
    return table


def target(df: pd.DataFrame) -> pd.Series:
    """y = 1 se dengue grave (CLASSI_FIN = 12) ou óbito pelo agravo (EVOLUCAO = 2) — D-02."""
    severe = code(df["CLASSI_FIN"]).isin({"12"})
    death = code(df["EVOLUCAO"]).isin({"2"})
    return (severe | death).astype("int8")


def _population_filters(df: pd.DataFrame) -> tuple[pd.Series, pd.Series, pd.Series]:
    """Os três filtros de D-45, na ordem do fluxograma: classificação, rótulo conhecido, idade."""
    classi = code(df["CLASSI_FIN"])
    classi_ok = classi.isin(CLASSI_DENGUE)
    # Grave (12) já tem y = 1 pela classificação; 10/11 precisam da evolução para saber do óbito.
    label_ok = code(df["EVOLUCAO"]).isin(EVOLUCAO_CONHECIDA) | (classi == "12")
    age = decode_age_years(df["NU_IDADE_N"])
    age_ok = age.between(0, 120)
    return classi_ok, label_ok, age_ok


def population_mask(df: pd.DataFrame) -> pd.Series:
    """População de D-45: dengue (10/11/12), rótulo conhecido e idade válida."""
    classi_ok, label_ok, age_ok = _population_filters(df)
    return classi_ok & label_ok & age_ok


def exclusion_flow(df: pd.DataFrame, year: str = "ANO") -> pd.DataFrame:
    """Fluxograma de exclusão (FR-02): linhas restantes após cada filtro, por ano."""
    classi_ok, label_ok, age_ok = _population_filters(df)
    steps = {
        "1. Total baixado": pd.Series(True, index=df.index),
        "2. CLASSI_FIN ∈ {10, 11, 12}": classi_ok,
        "3. Rótulo conhecido (12, ou 10/11 com EVOLUCAO ∈ {1, 2})": classi_ok & label_ok,
        "4. Idade válida (0–120 anos)": classi_ok & label_ok & age_ok,
    }
    flow = pd.DataFrame({name: mask.groupby(df[year]).sum() for name, mask in steps.items()}).T
    flow.columns.name = year
    return flow


def exclusion_reasons(df: pd.DataFrame, year: str = "ANO") -> dict[str, pd.DataFrame]:
    """Quem saiu em cada filtro, por valor do campo — para ver se casos graves saem por branco."""
    classi_ok, label_ok, age_ok = _population_filters(df)
    label_out = df[classi_ok & ~label_ok]
    age_out = df[classi_ok & label_ok & ~age_ok]
    return {
        "Removidos no filtro 2 (CLASSI_FIN)": counts_by_year(df[~classi_ok], "CLASSI_FIN", year),
        "Removidos no filtro 3 (rótulo), por EVOLUCAO": counts_by_year(label_out, "EVOLUCAO", year),
        "Removidos no filtro 3 (rótulo), por CLASSI_FIN": counts_by_year(label_out, "CLASSI_FIN",
                                                                        year),
        "Removidos no filtro 4 (idade), por CLASSI_FIN": counts_by_year(age_out, "CLASSI_FIN",
                                                                       year),
    }


def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    """Colunas derivadas usadas em vários passos: y, idade, faixa, região, gestante."""
    out = df.copy()
    out["y"] = target(out)
    out["idade_anos"] = decode_age_years(out["NU_IDADE_N"])
    out["faixa_etaria"] = pd.cut(out["idade_anos"], FAIXAS, right=False, labels=FAIXAS_ROTULOS)
    # Região pelo município de residência; se faltar, pela UF de residência.
    regiao = region_from_ibge(optional(out, "ID_MN_RESI")).astype("string")
    out["regiao"] = regiao.combine_first(region_from_ibge(optional(out, "SG_UF")).astype("string"))
    out["gestante"] = code(out["CS_GESTANT"]).isin(GESTANTE_SIM)
    return out


def prevalence_by_year(pop: pd.DataFrame, year: str = "ANO") -> pd.DataFrame:
    """Casos, eventos (y = 1) e prevalência por ano."""
    grouped = pop.groupby(year)["y"]
    table = pd.DataFrame({"casos": grouped.size(), "eventos (y=1)": grouped.sum()})
    table["prevalência (%)"] = 100 * table["eventos (y=1)"] / table["casos"]
    return table


def events_by(pop: pd.DataFrame, column: str) -> pd.DataFrame:
    """Casos e eventos por categoria, com alerta para < 100 eventos (§3.8)."""
    grouped = pop.groupby(pop[column].astype("string").fillna(BRANCO), observed=True)["y"]
    table = pd.DataFrame({"casos": grouped.size(), "eventos (y=1)": grouped.sum()})
    table["prevalência (%)"] = 100 * table["eventos (y=1)"] / table["casos"]
    table["< 100 eventos"] = np.where(table["eventos (y=1)"] < MIN_EVENTOS, "⚠️", "")
    table.index.name = column
    return table


def domain_counts(df: pd.DataFrame, columns: list[str], year: str = "ANO",
                  valid: tuple[str, ...] = ("1", "2")) -> pd.DataFrame:
    """Para campos 1/2: contagem de 1, 2, branco e outros valores, por campo e ano."""
    rows = []
    for column in columns:
        values = code(df[column])
        bucket = pd.Series(np.where(values.isin(valid), values, "outros"), index=df.index)
        bucket = bucket.where(values.notna(), BRANCO)
        counts = pd.crosstab(bucket, df[year])
        for value, row in counts.iterrows():
            rows.append({"campo": column, "valor": value, **row.to_dict()})
    return pd.DataFrame(rows).set_index(["campo", "valor"]).fillna(0).astype(int)


def other_values(df: pd.DataFrame, columns: list[str], valid: tuple[str, ...] = ("1", "2"),
                 top: int = 5) -> pd.DataFrame:
    """Quais valores fora do domínio aparecem (ex.: 9), para documentar a divergência."""
    rows = []
    for column in columns:
        values = code(df[column])
        outside = values[values.notna() & ~values.isin(valid)]
        for value, n in outside.value_counts().head(top).items():
            rows.append({"campo": column, "valor": value, "linhas": int(n)})
    return pd.DataFrame(rows, columns=["campo", "valor", "linhas"])


def absent_by_target(pop: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """% "Ausente" (branco ou ≠ 1/2) por campo, separado por y — check de vazamento (§3.8)."""
    rows = []
    for column in columns:
        absent = ~code(pop[column]).isin({"1", "2"})
        share = absent.groupby(pop["y"]).mean() * 100
        p0, p1 = share.get(0, np.nan), share.get(1, np.nan)
        rows.append({
            "campo": column,
            "% ausente y=0": p0,
            "% ausente y=1": p1,
            "razão y0/y1": p0 / p1 if p1 else np.nan,
        })
    table = pd.DataFrame(rows).set_index("campo")
    table["alerta (razão > 2)"] = np.where(table["razão y0/y1"] > 2, "⚠️", "")
    return table


def age_summary(df: pd.DataFrame, year: str = "ANO") -> pd.DataFrame:
    """Resumo da idade decodificada por ano: nulos, fora de [0, 120], quantis."""
    age = decode_age_years(df["NU_IDADE_N"])
    grouped = age.groupby(df[year])
    raw_blank = code(df["NU_IDADE_N"]).isna().groupby(df[year]).sum()
    return pd.DataFrame({
        "linhas": grouped.size(),
        "NU_IDADE_N em branco": raw_blank,
        "código inválido": age.isna().groupby(df[year]).sum() - raw_blank,
        "fora de [0, 120]": ((age < 0) | (age > 120)).groupby(df[year]).sum(),
        "mín": grouped.min(),
        "p50": grouped.median(),
        "p99": grouped.quantile(0.99),
        "máx": grouped.max(),
    })


def age_unit_counts(df: pd.DataFrame, year: str = "ANO") -> pd.DataFrame:
    """Distribuição do 1º dígito de NU_IDADE_N (unidade), para conferir a regra."""
    unit = code(df["NU_IDADE_N"]).str[0].fillna(BRANCO)
    table = pd.crosstab(unit, df[year])
    table.index.name = "unidade (1º dígito)"
    return table


def age_vs_birth_year(df: pd.DataFrame, tolerance_years: float = 1.0) -> pd.DataFrame:
    """Compara a idade decodificada com ano(DT_SIN_PRI) − ANO_NASC.

    Os microdados públicos não trazem DT_NASC, só o ano; sem dia e mês, a conta erra até
    1 ano para mais ou para menos, daí a tolerância padrão de 1 ano.
    """
    decoded = decode_age_years(df["NU_IDADE_N"])
    birth_year = pd.to_numeric(code(optional(df, "ANO_NASC")), errors="coerce")
    computed = parse_date(df["DT_SIN_PRI"]).dt.year - birth_year
    both = decoded.notna() & computed.notna()
    diff = (decoded - computed).abs()[both]
    return pd.DataFrame({
        "linhas com ANO_NASC e idade": [int(both.sum())],
        f"concordam (|dif| ≤ {tolerance_years:g} ano)": [int((diff <= tolerance_years).sum())],
        "% concordância": [100 * (diff <= tolerance_years).mean() if len(diff) else np.nan],
        "dif. p99 (anos)": [diff.quantile(0.99) if len(diff) else np.nan],
    })


def demography_inconsistencies(df: pd.DataFrame, year: str = "ANO") -> pd.DataFrame:
    """Gestante com sexo M e gestante com < 10 anos, por ano."""
    pregnant = code(df["CS_GESTANT"]).isin(GESTANTE_SIM)
    male = code(df["CS_SEXO"]) == "M"
    young = decode_age_years(df["NU_IDADE_N"]) < 10
    return pd.DataFrame({
        "gestante (1–4)": pregnant.groupby(df[year]).sum(),
        "gestante com sexo M": (pregnant & male).groupby(df[year]).sum(),
        "gestante com < 10 anos": (pregnant & young).groupby(df[year]).sum(),
    })


def ignored_share(df: pd.DataFrame, columns: list[str], ignored: set[str],
                  year: str = "ANO") -> pd.DataFrame:
    """% de ignorado (códigos dados) + branco por campo e ano (Q-05)."""
    rows = {}
    for column in columns:
        values = code(df[column])
        rows[column] = (values.isna() | values.isin(ignored)).groupby(df[year]).mean() * 100
    return pd.DataFrame(rows).T


def absent_share(df: pd.DataFrame, columns: list[str], year: str = "ANO") -> pd.DataFrame:
    """% ausente (branco ou ≠ 1/2) por campo e ano."""
    return pd.DataFrame({
        column: (~code(df[column]).isin({"1", "2"})).groupby(df[year]).mean() * 100
        for column in columns
    }).T


def alarm_filled_by_class(df: pd.DataFrame, alarm_columns: list[str]) -> pd.DataFrame:
    """% de linhas com algum ALRM_* preenchido, por valor de CLASSI_FIN (§3.6, M2)."""
    filled = pd.concat([code(df[c]).notna() for c in alarm_columns], axis=1).any(axis=1)
    classi = code(df["CLASSI_FIN"]).fillna(BRANCO)
    table = pd.DataFrame({
        "linhas": filled.groupby(classi).size(),
        "com algum ALRM_* preenchido": filled.groupby(classi).sum(),
    })
    table["%"] = 100 * table["com algum ALRM_* preenchido"] / table["linhas"]
    table.index.name = "CLASSI_FIN"
    return table


def notification_delay(df: pd.DataFrame, year: str = "ANO") -> pd.DataFrame:
    """Atraso sintomas → notificação, em dias, por ano (passo 8)."""
    onset, notified = parse_date(df["DT_SIN_PRI"]), parse_date(df["DT_NOTIFIC"])
    delay = (notified - onset).dt.days
    grouped = delay.groupby(df[year])
    return pd.DataFrame({
        "com as duas datas": grouped.count(),
        "sintomas > notificação": (delay < 0).groupby(df[year]).sum(),
        "mediana": grouped.median(),
        "p95": grouped.quantile(0.95),
        "p99": grouped.quantile(0.99),
        "máx": grouped.max(),
        "> 60 dias": (delay > 60).groupby(df[year]).sum(),
    })


def date_year_vs_file(df: pd.DataFrame, column: str, year: str = "ANO") -> pd.DataFrame:
    """Ano civil de uma data × ano do arquivo — define qual data marca o ano do split."""
    date_year = parse_date(df[column]).dt.year.astype("Int64").astype("string").fillna(BRANCO)
    table = pd.crosstab(date_year, df[year])
    table.index.name = f"ano de {column}"
    return table


def duplicates(pop: pd.DataFrame, year: str = "ANO") -> pd.DataFrame:
    """Duplicatas pela chave composta do roteiro (passo 9), com desfechos conflitantes.

    Chave: nascimento + sexo + município de residência + data dos sintomas. Nascimento usa
    DT_NASC se existir, senão ANO_NASC + idade (os públicos só têm o ano), senão a idade.
    """
    idade = "idade:" + code(pop["NU_IDADE_N"]).fillna(BRANCO)
    ano_nasc = code(optional(pop, "ANO_NASC"))
    birth = code(optional(pop, "DT_NASC")).fillna("ano:" + ano_nasc + "|" + idade).fillna(idade)
    key = (birth + "|" + code(pop["CS_SEXO"]).fillna(BRANCO) + "|"
           + code(optional(pop, "ID_MN_RESI")).fillna(BRANCO) + "|"
           + code(pop["DT_SIN_PRI"]).fillna(BRANCO))
    group_size = key.map(key.value_counts())
    in_dup = group_size > 1
    y_spread = pop["y"].groupby(key).transform("nunique")
    return pd.DataFrame({
        "linhas": pop.groupby(year).size(),
        "linhas em grupo duplicado": in_dup.groupby(pop[year]).sum(),
        "% em grupo duplicado": 100 * in_dup.groupby(pop[year]).mean(),
        "linhas excedentes (remoção manteria 1)": (in_dup & key.duplicated()).groupby(
            pop[year]).sum(),
        "linhas em grupos com y conflitante": (y_spread > 1).groupby(pop[year]).sum(),
    })


def cases_by_uf(pop: pd.DataFrame, population: pd.Series | None = None,
                year: str = "ANO") -> pd.DataFrame:
    """Casos confirmados por UF de residência e ano; por 100 mil hab. se houver população."""
    uf = code(pop["SG_UF"]).fillna(BRANCO)
    table = pd.crosstab(uf, pop[year])
    table.index.name = "SG_UF (residência)"
    if population is not None:
        rate = table.div(population.reindex(table.index), axis=0) * 100_000
        table = table.join(rate.add_prefix("por 100 mil — "))
    return table


def markdown_table(table: pd.DataFrame, decimals: int = 1) -> str:
    """Tabela Markdown sem dependências extras (tabulate)."""
    frame = table.reset_index() if not isinstance(table.index, pd.RangeIndex) else table

    def fmt(value) -> str:
        if isinstance(value, float | np.floating):
            return "" if np.isnan(value) else f"{value:,.{decimals}f}".replace(",", " ")
        if isinstance(value, int | np.integer):
            return f"{value:,}".replace(",", " ")
        return str(value)

    header = "| " + " | ".join(str(c) for c in frame.columns) + " |"
    rule = "|" + "|".join("---" for _ in frame.columns) + "|"
    body = ["| " + " | ".join(fmt(v) for v in row) + " |" for row in frame.itertuples(index=False)]
    return "\n".join([header, rule, *body])

## Passo 0 — Download, conversão e manifesto

Baixa `DENGBR{AA}.dbc` de `FINAIS` (ou `PRELIM`, se não houver final), registra hash do `.dbc`,
data de revisão no DataSUS, hash do Parquet, linhas e versões em `data/manifest.json`.

In [ ]:
import json
import time

import pandas as pd
import pyarrow.parquet as pq

SECOES: dict[str, str] = {}   # seção do RESULTADOS.md → Markdown
RESUMO: dict[str, dict] = {}  # linha da tabela-resumo → {resultado, esperado, obs}

manifest = {}
for ano in ANOS:
    t0 = time.time()
    entry = fetch_year(ano, DATA_DIR)
    manifest[ano] = entry
    print(f"{ano}: {entry['linhas']:,} linhas, {entry['colunas']} colunas, "
          f"{entry['status_datasus']}, revisão {entry['revisao_datasus_utc'][:10]} "
          f"({time.time() - t0:.0f} s)")

manifest_table = pd.DataFrame({
    ano: {
        "arquivo": e["arquivo"],
        "status no DataSUS": e["status_datasus"],
        "revisão no DataSUS (UTC)": e["revisao_datasus_utc"][:19],
        "baixado em (UTC)": e["baixado_em_utc"][:19],
        "linhas": e["linhas"],
        "linhas no cabeçalho DBF": e["linhas_no_cabecalho_dbf"],
        "colunas": e["colunas"],
        "SHA-256 .dbc": e["sha256_dbc"][:16] + "…",
        "SHA-256 Parquet": e["sha256_parquet"][:16] + "…",
    } for ano, e in manifest.items()
})
env = next(iter(manifest.values()))["ambiente"]
SECOES["0. Manifesto dos dados"] = (
    markdown_table(manifest_table) + "\n\nHashes completos em `data/manifest.json`. Ambiente: "
    + ", ".join(f"{k} {v}" for k, v in env.items()) + "."
)
cols_by_year = {ano: set(e["nomes_colunas"]) for ano, e in manifest.items()}
common = set.intersection(*cols_by_year.values())
only = {ano: sorted(c - common) for ano, c in cols_by_year.items() if c - common}
SECOES["0. Manifesto dos dados"] += (
    f"\n\nColunas em comum nos {len(ANOS)} anos: {len(common)}. "
    + ("Colunas exclusivas: " + "; ".join(f"{a}: {', '.join(c)}" for a, c in only.items())
       if only else "Mesmo schema em todos os anos.")
)
RESUMO["0"] = {
    "item": "Hash, data e versão de cada Parquet", "secao": "§3.14",
    "resultado": "; ".join(f"{a}: {e['linhas']:,} linhas, {e['status_datasus']}"
                           for a, e in manifest.items()),
    "esperado": "sim" if all(e["linhas"] == e["linhas_no_cabecalho_dbf"]
                             for e in manifest.values()) else "revisar",
    "obs": "Linhas lidas = cabeçalho do DBF." if all(
        e["linhas"] == e["linhas_no_cabecalho_dbf"] for e in manifest.values())
        else "Linhas lidas ≠ cabeçalho do DBF (registros apagados?).",
}
manifest_table

## Carga das colunas usadas

Só as colunas do roteiro, como `category`, para caber na RAM. `ALRM_*` e `GRAV_*` viram uma
flag "algum campo preenchido" já na carga (passo 7 e contingência do passo 1).

In [ ]:
import numpy as np

BASE_COLS = (CAMPOS_ALVO + COMORBIDADES + SINAIS
             + ["NU_IDADE_N", "ANO_NASC", "CS_SEXO", "CS_GESTANT", "CS_RACA", "CS_ESCOL_N",
                "SG_UF", "ID_MN_RESI", "SG_UF_NOT", "DT_NOTIFIC", "DT_SIN_PRI",
                "NDUPLIC_N", "TP_SISTEMA"])
# DT_NASC não é publicada nos microdados (anonimização); só ANO_NASC. Colunas ausentes
# aparecem em "Faltando" e as funções tratam como branco.
faltando = {}
partes = []
for ano in ANOS:
    path = DATA_DIR / manifest[ano]["parquet"]
    schema = pq.read_schema(path).names
    alarm = [c for c in schema if c.startswith("ALRM_")]
    grav = [c for c in schema if c.startswith("GRAV_")]
    usar = [c for c in BASE_COLS if c in schema]
    faltando[ano] = [c for c in BASE_COLS if c not in schema]
    tabela = pq.read_table(path, columns=usar + alarm + grav)
    parte = tabela.select(usar).to_pandas(strings_to_categorical=True)
    flags = tabela.select(alarm + grav).to_pandas()
    parte["ALRM_ANY"] = np.where(flags[alarm].notna().any(axis=1), "1", None) if alarm else None
    parte["GRAV_ANY"] = np.where(flags[grav].notna().any(axis=1), "1", None) if grav else None
    parte["ANO"] = ano
    partes.append(parte)
    del tabela, flags
    print(ano, "ALRM_*:", alarm)
    print(ano, "GRAV_*:", grav)

df = pd.concat(partes, ignore_index=True)
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].astype("category")
del partes
print("Faltando:", {a: c for a, c in faltando.items() if c} or "nenhuma")
print(f"{len(df):,} linhas carregadas")

## Passo 1 — Códigos do alvo (BLOQUEANTE)

In [ ]:
partes_md = []
for campo in CAMPOS_ALVO:
    t = counts_by_year(df, campo)
    partes_md.append(f"**{campo}**\n\n" + markdown_table(t))
    display(t)
SECOES["1. Códigos do alvo"] = "\n\n".join(partes_md)

classi_vals = set(code(df["CLASSI_FIN"]).dropna().unique())
tem_novo = {"10", "11", "12"} <= classi_vals
# 1 = confirmado e 2 = descartado só existem no esquema de 2024; 8 = inconclusivo aparece junto
# com 10/11/12 nos arquivos reais (profiling de 2026-11-03), então não indica troca de esquema.
tem_esquema_2024 = bool({"1", "2"} & classi_vals)
por_ano = {ano: sorted(set(code(df.loc[df.ANO == ano, "CLASSI_FIN"]).dropna().unique()))
           for ano in ANOS}
BLOQUEIO = (not tem_novo) or tem_esquema_2024
RESUMO["1"] = {
    "item": "Valores de CLASSI_FIN/EVOLUCAO/CRITERIO por ano", "secao": "§3.6",
    "resultado": "; ".join(f"{a}: CLASSI_FIN ∈ {{{', '.join(v)}}}" for a, v in por_ano.items()),
    "esperado": "não — PARAR" if BLOQUEIO else "sim",
    "obs": ("Apareceram códigos 1/2 ou faltam 10/11/12: D-02 precisa ser redecidido."
            if BLOQUEIO else "10/11/12 presentes e 1/2 ausentes; D-02 vale."),
}
print("BLOQUEIO" if BLOQUEIO else "OK — 10/11/12 presentes", por_ano)

grav_by_class = alarm_filled_by_class(df, ["GRAV_ANY"]).rename(
    columns={"com algum ALRM_* preenchido": "com algum GRAV_* preenchido"})
SECOES["1. Códigos do alvo"] += (
    "\n\n**Contingência: algum `GRAV_*` preenchido, por `CLASSI_FIN`** (todos os anos)\n\n"
    + markdown_table(grav_by_class))
grav_by_class

## Passo 2 — Fluxograma de exclusão (FR-02)

In [ ]:
flow = exclusion_flow(df)
reasons = exclusion_reasons(df)
SECOES["2. Fluxograma de exclusão"] = markdown_table(flow) + "".join(
    f"\n\n**{nome}**\n\n" + markdown_table(t) for nome, t in reasons.items())
final = flow.iloc[-1]
RESUMO["2"] = {
    "item": "Fluxograma de exclusão", "secao": "§3.6, FR-02",
    "resultado": "; ".join(f"{a}: {flow.loc[flow.index[0], a]:,} → {final[a]:,}" for a in ANOS),
    "esperado": "revisar", "obs": "Conferir motivos de saída no detalhe.",
}
display(flow)
for nome, t in reasons.items():
    print(nome); display(t)

## Passo 3 — Prevalência e número de eventos

In [ ]:
pop = add_derived(df[population_mask(df)])
prev = prevalence_by_year(pop)
dev, test = pop[pop.ANO.isin([2022, 2023])], pop[pop.ANO == 2024]
por_regiao_dev = events_by(dev, "regiao")
sub_2024 = {
    "faixa etária": events_by(test, "faixa_etaria"),
    "sexo": events_by(test.assign(sexo=code(test["CS_SEXO"])), "sexo"),
    "gestante (1–4)": events_by(test, "gestante"),
    "região": events_by(test, "regiao"),
}
SECOES["3. Prevalência e eventos"] = (
    "**Prevalência por ano**\n\n" + markdown_table(prev, 2)
    + "\n\n**Eventos por região — desenvolvimento 2022–23 (dobras leave-one-region-out)**\n\n"
    + markdown_table(por_regiao_dev, 2)
    + "".join(f"\n\n**Teste 2024 — por {k}**\n\n" + markdown_table(v, 2)
              for k, v in sub_2024.items())
)
alertas = [f"{k}={i}" for k, v in sub_2024.items() for i in v.index[v["< 100 eventos"] != ""]]
alertas_dev = list(por_regiao_dev.index[por_regiao_dev["< 100 eventos"] != ""])
RESUMO["3a"] = {"item": "Prevalência de y por ano", "secao": "§3.7",
                "resultado": "; ".join(f"{a}: {r['prevalência (%)']:.2f}% "
                                       f"({int(r['eventos (y=1)']):,} eventos)"
                                       for a, r in prev.iterrows()),
                "esperado": "revisar", "obs": ""}
RESUMO["3b"] = {"item": "Eventos por região (2022–23)", "secao": "§3.8",
                "resultado": "mín. " + f"{int(por_regiao_dev['eventos (y=1)'].min()):,} eventos",
                "esperado": "não" if alertas_dev else "sim",
                "obs": ("< 100 eventos: " + ", ".join(map(str, alertas_dev))) if alertas_dev
                else "Todas as regiões com ≥ 100 eventos."}
RESUMO["3c"] = {"item": "Eventos por subgrupo (2024)", "secao": "§3.8, §3.12",
                "resultado": f"{len(alertas)} subgrupo(s) com < 100 eventos",
                "esperado": "não" if alertas else "sim",
                "obs": ", ".join(alertas)}
display(prev); display(por_regiao_dev)
for k, v in sub_2024.items():
    print(k); display(v)

## Passo 4 — Comorbidades e o check de vazamento

In [ ]:
comorb_counts = domain_counts(df, COMORBIDADES)
comorb_outros = other_values(df, COMORBIDADES)
leak = absent_by_target(pop, COMORBIDADES)
SECOES["4. Comorbidades"] = (
    "**Valores por ano (todas as notificações)**\n\n" + markdown_table(comorb_counts)
    + "\n\n**Valores fora de 1/2 mais comuns**\n\n"
    + (markdown_table(comorb_outros) if len(comorb_outros) else "Nenhum.")
    + "\n\n**% Ausente × alvo (população final)**\n\n" + markdown_table(leak, 2)
)
pct_ausente = 100 * (~code(pop[COMORBIDADES[0]]).isin({"1", "2"})).mean()
RESUMO["4a"] = {"item": "Valores das comorbidades", "secao": "§3.7",
                "resultado": f"{COMORBIDADES[0]}: {pct_ausente:.1f}% ausente na população final",
                "esperado": "revisar",
                "obs": "Valores fora de 1/2: " + (", ".join(sorted(set(comorb_outros['valor'])))
                                                  if len(comorb_outros) else "nenhum")}
flagged = list(leak.index[leak["alerta (razão > 2)"] != ""])
RESUMO["4b"] = {"item": "% ausente × y (vazamento)", "secao": "§3.8",
                "resultado": f"razão y0/y1 entre {leak['razão y0/y1'].min():.2f} e "
                             f"{leak['razão y0/y1'].max():.2f}",
                "esperado": "não" if flagged else "sim",
                "obs": ("Razão > 2: " + ", ".join(flagged) + " → Q-06") if flagged else
                "Nenhuma comorbidade com razão > 2."}
display(comorb_counts); display(comorb_outros); display(leak)

## Passo 5 — Idade

In [ ]:
age = age_summary(df)
units = age_unit_counts(df)
concord = age_vs_birth_year(df)
SECOES["5. Idade"] = (
    markdown_table(age, 2) + "\n\n**Unidade (1º dígito de NU_IDADE_N)**\n\n"
    + markdown_table(units)
    + "\n\n**Idade decodificada × (ano de DT_SIN_PRI − ANO_NASC)** — tolerância de 1 ano, "
    + "porque os microdados não trazem DT_NASC, só o ano de nascimento"
    + "\n\n" + markdown_table(concord, 2)
)
fora = int(age["fora de [0, 120]"].sum())
RESUMO["5"] = {"item": "Idade: decodificação e faixa", "secao": "§3.7, §3.9",
               "resultado": f"{concord['% concordância'].iloc[0]:.1f}% concordam com "
                            "ANO_NASC (±1 ano); "
                            f"{fora} fora de [0, 120]; "
                            f"{int(age['código inválido'].sum()):,} códigos inválidos",
               "esperado": "sim" if concord["% concordância"].iloc[0] > 95 and fora == 0
               else "revisar", "obs": ""}
display(age); display(units); display(concord)

## Passo 6 — Demografia

In [ ]:
demo = {c: counts_by_year(df, c) for c in ["CS_SEXO", "CS_GESTANT", "CS_RACA"]}
incons = demography_inconsistencies(df)
ign = ignored_share(df, ["CS_RACA", "CS_ESCOL_N"], ignored={"9"})
SECOES["6. Demografia"] = "".join(f"**{c}**\n\n" + markdown_table(t) + "\n\n"
                                  for c, t in demo.items()) + (
    "**Inconsistências**\n\n" + markdown_table(incons)
    + "\n\n**% ignorado (9) + branco — Q-05**\n\n" + markdown_table(ign, 1))
RESUMO["6"] = {"item": "Demografia e inconsistências", "secao": "§3.7",
               "resultado": f"gestante com sexo M: {int(incons['gestante com sexo M'].sum()):,}; "
                            f"raça ignorada/branco: "
                            + ", ".join(f"{a} {ign.loc['CS_RACA', a]:.0f}%" for a in ANOS),
               "esperado": "revisar", "obs": "Ver Q-05 (raça/escolaridade)."}
for c, t in demo.items():
    print(c); display(t)
display(incons); display(ign)

## Passo 7 — Sinais clínicos e de alarme

In [ ]:
sinais = absent_share(df, [c for c in SINAIS if c in df.columns])
alarm = alarm_filled_by_class(df, ["ALRM_ANY"])
SECOES["7. Sinais clínicos e de alarme"] = (
    "**% ausente (branco ou ≠ 1/2) por ano**\n\n" + markdown_table(sinais, 1)
    + "\n\n**Algum `ALRM_*` preenchido, por `CLASSI_FIN`** (todos os anos)\n\n"
    + markdown_table(alarm, 1))
RESUMO["7a"] = {"item": "Sinais clínicos: % ausente", "secao": "§3.8",
                "resultado": f"entre {sinais.min().min():.1f}% e {sinais.max().max():.1f}%",
                "esperado": "revisar", "obs": ""}
pct10 = alarm.loc["10", "%"] if "10" in alarm.index else float("nan")
pct11 = alarm.loc["11", "%"] if "11" in alarm.index else float("nan")
# A restrição de M2 vale enquanto o preenchimento separar as classes (revela a classificação).
revela_classe = pct11 - pct10 > 50
RESUMO["7b"] = {"item": "ALRM_* preenchido por CLASSI_FIN", "secao": "§3.6",
                "resultado": f"algum ALRM_* preenchido: {pct10:.1f}% em CLASSI_FIN=10, "
                             f"{pct11:.1f}% em 11",
                "esperado": "sim" if revela_classe else "revisar",
                "obs": "Preenchimento revela a classe: restrição de M2 a 11/12 necessária."
                if revela_classe else "Preenchimento parecido entre classes: rever M2 (§3.6)."}
display(sinais); display(alarm)

## Passo 8 — Datas

In [ ]:
delay = notification_delay(df)
ano_sint = date_year_vs_file(df, "DT_SIN_PRI")
ano_notif = date_year_vs_file(df, "DT_NOTIFIC")
SECOES["8. Datas"] = (
    "**Atraso sintomas → notificação (dias)**\n\n" + markdown_table(delay, 1)
    + "\n\n**Ano de DT_SIN_PRI × ano do arquivo**\n\n" + markdown_table(ano_sint)
    + "\n\n**Ano de DT_NOTIFIC × ano do arquivo**\n\n" + markdown_table(ano_notif))
RESUMO["8"] = {"item": "Atraso sintomas → notificação", "secao": "§3.9",
               "resultado": "; ".join(f"{a}: mediana {r['mediana']:.0f} d, p99 {r['p99']:.0f} d, "
                                      f"{int(r['> 60 dias']):,} > 60 d"
                                      for a, r in delay.iterrows()),
               "esperado": "revisar", "obs": "Ver também ano da data × ano do arquivo."}
display(delay); display(ano_sint); display(ano_notif)

## Passo 9 — Duplicatas

In [ ]:
dups = duplicates(pop)
nduplic = counts_by_year(pop, "NDUPLIC_N") if "NDUPLIC_N" in pop else None
SECOES["9. Duplicatas"] = (
    "Chave: ANO_NASC + idade (os microdados não trazem DT_NASC) + CS_SEXO + ID_MN_RESI + "
    "DT_SIN_PRI, na população final. Sem a data completa de nascimento, a chave é mais fraca "
    "e tende a superestimar duplicatas.\n\n" + markdown_table(dups, 2)
    + ("\n\n**NDUPLIC_N (marcação de duplicidade do próprio SINAN), população final**\n\n"
       + markdown_table(nduplic) if nduplic is not None else ""))
RESUMO["9"] = {"item": "Duplicatas", "secao": "§3.6",
               "resultado": "; ".join(f"{a}: {r['% em grupo duplicado']:.2f}% em grupo duplicado"
                                      for a, r in dups.iterrows()),
               "esperado": "revisar",
               "obs": f"{int(dups['linhas em grupos com y conflitante'].sum()):,} linhas em grupos "
                      "com desfecho conflitante → Q-04. Chave sem DT_NASC (mais fraca)."}
display(dups)
if nduplic is not None:
    display(nduplic)

## Passo 10 — Cobertura do Espírito Santo

População: Censo 2022 do IBGE (SIDRA, tabela 4709), usada para os três anos. Se a API falhar,
a tabela sai só com contagens.

In [ ]:
import urllib.request

UF_SIGLA = {"11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
            "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL",
            "28": "SE", "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP", "41": "PR",
            "42": "SC", "43": "RS", "50": "MS", "51": "MT", "52": "GO", "53": "DF"}
try:
    url = "https://apisidra.ibge.gov.br/values/t/4709/n3/all/v/93/p/2022"
    with urllib.request.urlopen(url, timeout=60) as r:
        rows = json.load(r)[1:]
    populacao = pd.Series({row["D1C"]: int(row["V"]) for row in rows})
    fonte_pop = "IBGE, Censo 2022 (SIDRA 4709)"
except Exception as exc:  # sem internet ou API fora do ar
    populacao, fonte_pop = None, f"indisponível ({type(exc).__name__})"

uf = cases_by_uf(pop, populacao)
sistema = None
if "TP_SISTEMA" in pop:
    sistema = pd.crosstab(code(pop["SG_UF"]).map(UF_SIGLA).fillna(BRANCO),
                          code(pop["TP_SISTEMA"]).fillna(BRANCO))
    sistema = sistema.loc[[i for i in ["ES", "MG", "RJ", "BA"] if i in sistema.index]]
uf.index = [f"{i} ({UF_SIGLA.get(i, '?')})" for i in uf.index]
foco = uf[uf.index.str.contains(r"\((?:ES|MG|RJ|BA)\)")]
SECOES["10. Cobertura do Espírito Santo"] = (
    f"População: {fonte_pop}.\n\n**ES e vizinhos**\n\n" + markdown_table(foco, 1)
    + ("\n\n**TP_SISTEMA (sistema de origem) — ES e vizinhos, 2022–24**\n\n"
       + markdown_table(sistema) if sistema is not None else "")
    + "\n\n**Todas as UFs**\n\n" + markdown_table(uf, 1))
if populacao is not None:
    taxa = foco[[c for c in foco.columns if str(c).startswith("por 100 mil")]]
    es = taxa[taxa.index.str.contains("(ES)", regex=False)].iloc[0]
    viz = taxa[~taxa.index.str.contains("(ES)", regex=False)].median()
    res10 = "; ".join(f"{c[-4:]}: ES {es[c]:.0f} vs mediana vizinhos {viz[c]:.0f} /100 mil"
                      for c in taxa.columns)
else:
    res10 = "sem população; ver contagens"
RESUMO["10"] = {"item": "Cobertura do ES", "secao": "§3.7", "resultado": res10,
                "esperado": "revisar", "obs": f"População: {fonte_pop}."}
foco

## Passo 11 — Memória

In [ ]:
import os

try:
    import psutil
    ram_total = psutil.virtual_memory().total / 2**30
    ram_livre = psutil.virtual_memory().available / 2**30
except ImportError:
    ram_total = ram_livre = float("nan")
mem_df = df.memory_usage(deep=True).sum() / 2**30
mem_pop = pop.memory_usage(deep=True).sum() / 2**30
SECOES["11. Memória"] = markdown_table(pd.DataFrame({
    "item": ["RAM total da sessão (GiB)", "RAM disponível agora (GiB)",
             f"DataFrame carregado ({len(df):,} linhas × {df.shape[1]} colunas, GiB)",
             f"População final ({len(pop):,} linhas, GiB)"],
    "valor": [ram_total, ram_livre, mem_df, mem_pop],
}).set_index("item"), 2) + f"\n\nAmbiente: {'Colab' if IN_COLAB else 'local'}."
RESUMO["11"] = {"item": "RAM e tamanho do DataFrame", "secao": "NFR-01",
                "resultado": f"RAM {ram_total:.1f} GiB; DataFrame {mem_df:.2f} GiB "
                             f"({'Colab' if IN_COLAB else 'local'})",
                "esperado": "sim" if mem_df < 0.25 * ram_total else "revisar", "obs": ""}

## Gera `RESULTADOS.md`

In [ ]:
from datetime import datetime, timezone

ordem = ["0", "1", "2", "3a", "3b", "3c", "4a", "4b", "5", "6", "7a", "7b", "8", "9", "10", "11"]
linhas = ["| # | Item | Seção do spec | Resultado (número ou achado) | Dentro do esperado? "
          "| Observação |", "|---|---|---|---|---|---|"]
for k in ordem:
    r = RESUMO.get(k)
    if r:
        linhas.append(f"| {k} | {r['item']} | {r['secao']} | {r['resultado']} | {r['esperado']} "
                      f"| {r['obs']} |")
gerado = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
texto = "\n".join([
    "# Resultados do profiling — SINAN Dengue 2022–2024",
    "",
    f"> Gerado automaticamente por `profiling/01_profiling.ipynb` em {gerado} "
    f"({'Colab' if IN_COLAB else 'local'}). Não edite à mão: rode o notebook de novo.",
    "> \"Dentro do esperado?\" = `sim`/`não` quando a regra é objetiva; `revisar` quando "
    "depende de julgamento.",
    "",
    "## Resumo",
    "",
    *linhas,
    "",
    *[f"## {titulo}\n\n{conteudo}\n" for titulo, conteudo in SECOES.items()],
])
OUT_MD.write_text(texto, encoding="utf-8")
print("Gravado:", OUT_MD)